# Exercise 7 — Neural network from scratch

Today there is no library model to call. We build a neural network out of
NumPy arrays: one neuron first, then a two-layer network whose
**backpropagation you write out yourself**, exactly as the lecture on
artificial neural networks derives it. You rewrite it with a loop over the
layers, so that it can have any number of layers, and then turn your own code
into PyTorch one step at a time, checking after every step that the numbers
stay the same. Finally an agent grows your own loop-based network into a
deeper one that classifies microscope images of blood cells.

## When to use AI agents

- **Tasks 1–5: work without AI agents.** Write and understand the code yourself. Use the formulas, the step comments in every cell and teacher help.
- **Task 6: use your coding agent.** Have it extend **your own network from Task 4** into a deeper classifier, run the experiments and generate the report.
- **Your review is your own work.** Check the agent's output and write `REVIEW.md` yourself; do not ask the agent to write it.

| Task | Your output |
| --- | --- |
| 1. One neuron by hand | A forward pass, a gradient and one update on paper |
| 2. One neuron in NumPy | Your SGD loop, compared with the closed form; a numerical gradient |
| 3. A two-layer network | One plain script: forward pass, loss, backward pass and update in a mini-batch loop |
| 4. Any number of layers | Your script as functions with a loop over the layers — the same numbers for two layers, then a deeper network |
| 5. From your code to PyTorch | Autograd instead of your backward pass, an optimizer instead of your update — the same numbers again |
| 6. Agent extension | Your network, deeper, on blood-cell images — the compulsory hand-in |

Everything follows the lecture notation. For one neuron

$$a = \mathbf w^T \mathbf x + b, \qquad \hat y = f(a).$$

**Write the code yourself.** Data generation, plotting and the checks are
supplied; every cell you complete has short step comments. Where your code
goes, the cell says `# TODO: your code here`, often followed by placeholder
lines such as `L = np.nan` that only keep the notebook running —
**replace them with your own code**. Complete solutions are in the teacher
version only. Start Task 6 in class and finish it at home, within **48 hours
after your lab**.

<p style="color: #c00000;"><strong>! Compulsory hand-in: complete and submit Task 6 within 48 hours after your lab. Attendance will not be credited unless you complete this requirement.</strong></p>

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

## Task 1 — One neuron by hand

**Close the laptop for this part.** Everything below fits on the blackboard,
and doing it once by hand is the only way to see that backpropagation is
nothing but the chain rule applied systematically.

One neuron with two inputs and the squared-error loss. The bias is left out
so that the arithmetic stays short — and it costs us nothing, because a bias
is just a weight on a constant input $x_0 = 1$, as the lecture notes.

$$\begin{array}{ll}
\text{inputs} & \mathbf x = (x_1, x_2) = (1,\ 2) \\
\text{weights} & \mathbf w = (w_1, w_2) = (0.5,\ -0.25) \\
\text{target} & y = 1 \\
\text{loss} & L = \tfrac12 (\hat y - y)^2 \\
\text{learning rate} & \eta = 1
\end{array}$$

Compute, in this order:

1. the inner potential $a = w_1 x_1 + w_2 x_2$ (the numbers are chosen so that
   it comes out nice);
2. for the **linear** activation $f(a) = a$, $f'(a) = 1$: the prediction
   $\hat y$, the loss $L$, and the gradient
   $\nabla_{\mathbf w} L = (\hat y - y)\, f'(a)\, \mathbf x$;
3. the same for the **sigmoid** activation
   $f(a) = \sigma(a) = \dfrac{1}{1 + e^{-a}}$, whose derivative is
   $f'(a) = f(a)\,\big(1 - f(a)\big)$;
4. one gradient step of the sigmoid neuron,
   $\mathbf w \leftarrow \mathbf w - \eta \nabla_{\mathbf w} L$, and then the
   new $a$, the new $\hat y$ and the new $L$. Did the loss go down?

Finally, answer in one sentence: the input, the target and the loss function
are the same in both columns, so **why** is the sigmoid gradient eight times
smaller than the linear one? Compare the three factors.

*Your numbers and the one-sentence answer:*

Now the same numbers in code. Nothing here is a library call — you are
writing the formulas you have just used on the blackboard, and the printout
must reproduce your paper.

In [ ]:
def sigmoid(a):
    return 1.0 / (1.0 + np.exp(-a))


x = np.array([1.0, 2.0])
w = np.array([0.5, -0.25])
y = 1.0
eta = 1.0

a = float(x @ w)

# 1. Linear neuron: the prediction, the loss and the gradient (y_hat_lin, loss_lin, grad_lin).
# 2. Sigmoid neuron: the same three quantities with the sigmoid activation.
# 3. One gradient step of the sigmoid neuron gives w_new.
# 4. Recompute a_new, y_hat_new and loss_new with w_new.
# TODO: your code here
y_hat_lin, loss_lin, grad_lin = 0.0, 0.0, np.zeros(2)
y_hat_sig, loss_sig, grad_sig = 0.0, 0.0, np.zeros(2)
w_new, a_new, y_hat_new, loss_new = w.copy(), 0.0, 0.0, 0.0

print(f"a = {a}")
print(f"linear : y_hat = {y_hat_lin:.4f}  L = {loss_lin:.4f}  grad = {np.round(grad_lin, 4)}")
print(f"sigmoid: y_hat = {y_hat_sig:.4f}  L = {loss_sig:.4f}  grad = {np.round(grad_sig, 4)}")
print()
print(f"after one step: w = {np.round(w_new, 4)}, a = {a_new:.4f}, "
      f"y_hat = {y_hat_new:.4f}, L = {loss_new:.4f}")
print(f"the loss went from {loss_sig:.4f} to {loss_new:.4f}")

## Task 2 — One neuron in NumPy

A neuron with a **linear** activation and one input is exactly the model
$\hat y = w x + b$ — linear regression. That is a good thing: we know what the
right answer is, so we can check whether gradient descent found it.

With the loss $L = \tfrac12 (\hat y - y)^2$ for a single sample, the chain
rule gives

$$\frac{dL}{dw} = (\hat y - y)\, x, \qquad \frac{dL}{db} = (\hat y - y),$$

because $f'(a) = 1$ for the linear activation and $da/db = 1$. We use
**stochastic gradient descent** (SGD): one sample, one update, and the order
of the samples is reshuffled at the start of every epoch, as the lecture
prescribes. One *epoch* is one pass over all 100 samples.

In [ ]:
np.random.seed(0)
X_lin = 5 * np.random.rand(100)
y_lin = X_lin * 0.5 + 2 + 0.4 * np.random.randn(100)

plt.plot(X_lin, y_lin, ".")
plt.xlabel("x")
plt.ylabel("y")
plt.title("100 samples from y = 0.5 x + 2 + noise")
plt.show()

**Write the whole training loop.** The starting point, the learning rate and
the random generator are set; you write the epochs, the shuffling, the
per-sample update and the record of the loss. Use the generator exactly as the
step comments say — then your numbers match everybody else's: after 200
epochs about $w = 0.495$, $b = 2.07$.

In [ ]:
w, b = 0.1, -0.1          # starting point, deliberately far from the answer
eta = 0.001               # learning rate
n_epochs = 200

rng = np.random.default_rng(0)
loss_curve = []           # the mean loss over all 100 samples after every epoch

# 1. Loop over n_epochs epochs; at the start of each, draw the order rng.permutation(len(X_lin)).
# 2. For every index i in that order: the forward pass and the error of sample i.
# 3. Update both parameters with the two derivatives above.
# 4. After each epoch append the mean loss over all samples to loss_curve.
# TODO: your code here
...

print(f"w = {w:.4f}, b = {b:.4f}")

**Did it find the right answer?** For linear regression we do not have to
guess: the minimum of the squared error has a closed form, and
`np.linalg.lstsq` gives it in one line. This is the same habit as in the
earlier labs — whenever a reference solution exists, compare against it.

In [ ]:
design = np.column_stack([X_lin, np.ones_like(X_lin)])
w_exact, b_exact = np.linalg.lstsq(design, y_lin, rcond=None)[0]

print(f"gradient descent : w = {w:.4f}, b = {b:.4f}")
print(f"closed form      : w = {w_exact:.4f}, b = {b_exact:.4f}")
print(f"loss of our neuron  : {np.mean(0.5 * (w * X_lin + b - y_lin) ** 2):.6f}")
print(f"loss of the optimum : {np.mean(0.5 * (w_exact * X_lin + b_exact - y_lin) ** 2):.6f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(X_lin, y_lin, ".")
grid = np.linspace(0, 5, 100)
axes[0].plot(grid, w * grid + b, "r-", label=f"y = {w:.3f} x + {b:.3f}")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].legend()
axes[1].plot(loss_curve)
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("mean loss")
axes[1].set_yscale("log")
plt.tight_layout()
plt.show()

The parameters are close but not identical, and the loss is a hair above the
optimum. That is not a bug: SGD with a **constant** learning rate never comes
to rest — every sample pulls the line a little in its own direction, so the
parameters keep jittering in a small neighbourhood of the minimum. Decaying
the learning rate is the standard fix (the lecture mentions it under SGD).

### The numerical gradient

The numerical gradient nudges **one** parameter by $\varepsilon$ up and down,
keeps all the others fixed and measures the slope of the loss:

$$\frac{dL}{dw} \approx \frac{L(w + \varepsilon) - L(w - \varepsilon)}{2\varepsilon}.$$

It cannot train a network: every parameter costs two forward passes, so one
step of a network with a million parameters would need two million forward
passes instead of one forward and one backward pass. What it is for is a
**check**: on a small network it is the referee of your analytic gradient —
if the two disagree, the gradient is wrong, however nicely the loss goes down
(the slides "Numerical gradient" of the lecture). Here the loss is the
**mean** over all 100 samples, so the analytic gradient is the mean of the
per-sample gradients.

In [ ]:
def total_loss(w, b):
    return np.mean(0.5 * (w * X_lin + b - y_lin) ** 2)


w_test, b_test = 0.3, 1.0
eps = 1e-6

# 1. The error of every sample at (w_test, b_test).
# 2. Analytic gradients of the mean loss with respect to w and b.
# 3. Central differences with total_loss, one parameter at a time.
# TODO: your code here
grad_w_analytic = grad_b_analytic = np.nan
grad_w_numeric = grad_b_numeric = np.nan

print(f"dL/dw  analytic {grad_w_analytic: .8f}   numeric {grad_w_numeric: .8f}")
print(f"dL/db  analytic {grad_b_analytic: .8f}   numeric {grad_b_numeric: .8f}")
print(f"largest difference: {max(abs(grad_w_analytic - grad_w_numeric), abs(grad_b_analytic - grad_b_numeric)):.2e}")

## Task 3 — A two-layer network, written out as a script

One neuron can only draw a straight line. To fit something curved we need a
**hidden layer**: many neurons with a nonlinear activation, whose outputs are
then combined by a single output neuron.

The network of this task, in the notation of the lecture ($\mathbf h_0$ is
the input, $\mathbf a_k$ the inner potentials of layer $k$, $\mathbf h_k$ its
activations). The lecture hides the bias in $W_k$ as a weight on a constant
input; here it is a separate array $\mathbf b_k$ — the same thing written out:

$$\mathbf a_1 = W_1 \mathbf h_0 + \mathbf b_1, \qquad \mathbf h_1 = f(\mathbf a_1)
  \quad (\text{ReLU}),$$
$$\mathbf a_2 = W_2 \mathbf h_1 + \mathbf b_2, \qquad \hat{\mathbf y} = \mathbf a_2
  \quad (\text{linear output, because this is regression}).$$

We keep **one sample per column**, so a mini-batch of $n$ samples is a matrix
with $n$ columns and the whole batch goes through in one matrix product. With
$m$ hidden neurons:

$$\begin{array}{l|l}
\text{array} & \text{shape} \\ \hline
\mathbf h_0 \ \text{(input)} & 1 \times n \\
W_1,\ \mathbf b_1 & m \times 1,\ \ m \times 1 \\
\mathbf a_1,\ \mathbf h_1 & m \times n \\
W_2,\ \mathbf b_2 & 1 \times m,\ \ 1 \times 1 \\
\hat{\mathbf y} & 1 \times n
\end{array}$$

(Most libraries, PyTorch included, use the opposite convention, one sample per
*row*; the formulas are then transposed.)

You write the whole network as **one plain script**: the forward pass, the
loss, the backward pass and the update, line by line, with no functions. It
is the most readable form of a neural network there is — every line is one
formula of the lecture.

The data are 150 noisy samples of $\sin x$; every tenth sample gets a large
random offset added on top, so the set also contains outliers. It is a curve
that a single neuron has no chance of fitting.

In [ ]:
np.random.seed(1)
X = 5 * np.random.rand(150)
Y = np.sin(X).ravel()
Y[::10] += 3 * (0.5 - np.random.rand(15))
Y = Y + 0.1 * np.random.randn(150)

plt.plot(X, Y, "*")
plt.xlabel("x")
plt.ylabel("y")
plt.title("150 samples: sin(x) + noise, every tenth an outlier")
plt.show()

The hidden layer uses ReLU; its derivative is what the backward pass needs:

$$f(\alpha) = \max(0, \alpha), \qquad
  f'(\alpha) = \begin{cases} 0 & \alpha < 0 \\ 1 & \alpha > 0 \end{cases}$$

(At $\alpha = 0$ the derivative does not exist; we use $0$ and nobody minds.)
In NumPy, ReLU is `np.maximum(a1, 0.0)` and its derivative is the mask
`(a1 > 0)`.

The four parameter arrays start as small random numbers: normally distributed
with mean 0 and standard deviation 0.1, in the shapes of the table above. Draw
them from the generator exactly as the step comments say, so that everybody
starts from the same network.

In [ ]:
m = 128                    # hidden neurons
gen = np.random.default_rng(0)

# 1. Draw W1_start, b1_start, W2_start and b2_start, in this order, each with
#    gen.normal(0.0, 0.1, shape) and the shape from the table (with m = 128).
# TODO: your code here
W1_start, b1_start = np.full((m, 1), np.nan), np.full((m, 1), np.nan)
W2_start, b2_start = np.full((1, m), np.nan), np.full((1, 1), np.nan)
print("W1", W1_start.shape, " b1", b1_start.shape, " W2", W2_start.shape, " b2", b2_start.shape)

### 3a. One mini-batch: forward, loss, backward

The loss is the **mean** squared error over the mini-batch,
$L = \frac{1}{n}\sum_{i=1}^{n} \tfrac12 (\hat y_i - y_i)^2$, so
$\nabla_{\hat{\mathbf y}} L = (\hat{\mathbf y} - \mathbf y)/n$. The backward pass
runs from the last layer down; with the names of the step comments it is

- `grad_a2` $= (\hat{\mathbf y} - \mathbf y)/n$ (the output is linear, $f'(\mathbf a_2) = 1$)
- `grad_W2` = `grad_a2` $\cdot\, \mathbf h_1^T$
- `grad_b2` = `grad_a2` summed over the batch
- `grad_h1` = $W_2^T \cdot$ `grad_a2`
- `grad_a1` = `grad_h1` $\odot f'(\mathbf a_1)$
- `grad_W1` = `grad_a1` $\cdot\, \mathbf h_0^T$
- `grad_b1` = `grad_a1` summed over the batch

where $\odot$ is the element-wise product (`*` in NumPy, while `@` is the
matrix product).

Before looping, get **one** step right. Take the first mini-batch (the first
16 samples) and compute, in this order: the forward pass, the loss, and the
gradients of all four parameters. Write the shape of every array into a
comment as you go — mistakes in backpropagation are almost always shape
mistakes, and **the shape of a gradient always equals the shape of the thing
it is a gradient of**.

In [ ]:
W1, b1, W2, b2 = W1_start, b1_start, W2_start, b2_start

# 1. The first mini-batch: the first 16 samples of X and Y, one sample per column (h0, y), and n.
# 2. Forward pass: a1, h1 (ReLU) and a2; the prediction y_hat is a2 (linear output).
# 3. The loss L: the mean over the batch, as defined above.
# 4. Output layer: grad_a2 (the loss is a mean, so divide by n), then grad_W2 and
#    grad_b2 (a bias gradient is summed over the batch: axis=1, keepdims=True).
# 5. Back through W2 to grad_h1, through the ReLU to grad_a1, then grad_W1 and grad_b1.
# TODO: your code here
L = np.nan
grad_W1, grad_b1 = np.full_like(W1, np.nan), np.full_like(b1, np.nan)
grad_W2, grad_b2 = np.full_like(W2, np.nan), np.full_like(b2, np.nan)

print(f"loss on the first mini-batch: {L:.6f}")
for name, grad, param in [("W1", grad_W1, W1), ("b1", grad_b1, b1),
                          ("W2", grad_W2, W2), ("b2", grad_b2, b2)]:
    assert grad.shape == param.shape, f"the gradient of {name} has shape {grad.shape}, {name} has {param.shape}"
print("every gradient has the shape of its parameter")

**Check the backward pass before training with it.** A wrong gradient does
not raise an error — it just trains badly. The supplied cell compares your
gradients with the numerical gradient of Task 2 on all 385 parameters: around
$10^{-10}$ is the accuracy of the finite difference itself; anything above
$10^{-6}$ means a bug.

In [ ]:
#@title Gradient check: your gradients against finite differences
def loss_of(W1, b1, W2, b2):
    # the loss of the first mini-batch, for any values of the parameters
    h0_check, y_check = X[:16].reshape(1, -1), Y[:16].reshape(1, -1)
    return np.mean(0.5 * (W2 @ np.maximum(W1 @ h0_check + b1, 0.0) + b2 - y_check) ** 2)


eps = 1e-6
starting = [W1_start, b1_start, W2_start, b2_start]
yours = [grad_W1, grad_b1, grad_W2, grad_b2]
largest = 0.0
for k, name in enumerate(["W1", "b1", "W2", "b2"]):
    for index in np.ndindex(starting[k].shape):
        up = [p.copy() for p in starting]
        down = [p.copy() for p in starting]
        up[k][index] += eps
        down[k][index] -= eps
        numeric = (loss_of(*up) - loss_of(*down)) / (2 * eps)
        largest = max(largest, abs(numeric - yours[k][index]))
    print(f"{name}: checked {starting[k].size} values")
gradient_difference = float(largest) if np.all([np.all(np.isfinite(g)) for g in yours]) else np.nan
print(f"largest analytic-vs-numeric difference: {gradient_difference:.2e}")

### 3b. The training loop

Now put your lines from 3a into a loop. One **epoch** is one pass over the
shuffled data in mini-batches of 16 (the last batch of an epoch is shorter:
150 = 9 · 16 + 6). After each mini-batch every parameter takes one step
against its gradient, $W \leftarrow W - \eta \nabla_W L$, and the same for
every $\mathbf b$. After each epoch, record the mean loss over all 150
samples, so that you can see the training progress.

**Expected result** ($\eta = 0.01$, 300 epochs, the generator used exactly as
the step comments say): a loss of about 0.281 after the first epoch and 0.054
after the last, against 0.290 for the constant model "always predict the
mean".

In [ ]:
W1, b1, W2, b2 = W1_start.copy(), b1_start.copy(), W2_start.copy(), b2_start.copy()   # the start of 3a
eta = 0.01
n_epochs = 300
batch_size = 16
rng = np.random.default_rng(0)
loss_curve_script = []

# 1. Loop over the epochs; at the start of each, draw the order once: rng.permutation(len(X)).
# 2. Walk through the order in steps of batch_size; the batch h0 and y one sample per column, as in 3a.
# 3. Your forward pass and backward pass from 3a, then update W1, b1, W2 and b2.
# 4. After each epoch: the forward pass on all 150 samples and the mean loss, appended to loss_curve_script.
# TODO: your code here
loss_curve_script = [np.nan] * n_epochs

loss_curve_script = np.array(loss_curve_script)
baseline_loss = np.mean(0.5 * (Y - Y.mean()) ** 2)
print(f"loss after 1 epoch   : {loss_curve_script[0]:.4f}")
print(f"loss after 300 epochs: {loss_curve_script[-1]:.4f}")
print(f"loss of the constant model 'always predict the mean': {baseline_loss:.4f}")

In [ ]:
grid = np.linspace(X.min(), X.max(), 200).reshape(1, -1)
fit = (W2 @ np.maximum(W1 @ grid + b1, 0.0) + b2).ravel()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(X, Y, "*", markersize=4)
axes[0].plot(grid.ravel(), fit, "r-")
axes[0].set_ylim(-2.6, 2.6)
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].set_title(f"the fit after {len(loss_curve_script)} epochs")
axes[1].plot(loss_curve_script)
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("mean loss")
axes[1].set_yscale("log")
axes[1].set_title("training loss")
plt.tight_layout()
plt.show()

## Task 4 — A network with any number of layers

Your Task 3 script can be run only once, as it is, and it knows exactly two
layers: the names `W1`, `b1`, `W2`, `b2` are written into its lines. Training
it again with other settings needs **functions**, and a deeper network does
not need new mathematics —
it repeats the same steps once more per layer: the forward recipe from the
first layer to the last, the backward recipe from the last layer to the first
(the slide "General neural net — each layer and NN learning" of the lecture,
also on the exercise page). So instead of writing one more set of lines per
layer, you write each recipe **once, inside a loop over the layers**. This is
also the code the agent extends in Task 6.

A network is described by the **list of its layer sizes**: `[1, 128, 1]` is
the network of Task 3, `[1, 32, 32, 1]` a network with two hidden
layers. It has $K$ = `len(sizes) - 1` layers, and layer $k$ ($k = 1, \dots, K$)
has the arrays

$$\begin{array}{l|l}
\text{array} & \text{shape} \\ \hline
W_k & \text{sizes}[k] \times \text{sizes}[k-1] \\
\mathbf b_k & \text{sizes}[k] \times 1 \\
\mathbf a_k,\ \mathbf h_k & \text{sizes}[k] \times n
\end{array}$$

**Forward**, for $k = 1, \dots, K$, starting from the input $\mathbf h_0$:

$$\mathbf a_k = W_k \mathbf h_{k-1} + \mathbf b_k, \qquad \mathbf h_k = f(\mathbf a_k),$$

with ReLU in the hidden layers and the identity in the last layer, so the
prediction is $\hat{\mathbf y} = \mathbf h_K = \mathbf a_K$.

**Backward**, for $k = K, \dots, 1$, starting from
$\nabla_{\mathbf a_K} L = (\hat{\mathbf y} - \mathbf y)/n$ (the output layer
is linear):

$$\nabla_{W_k} L = \nabla_{\mathbf a_k} L \; \mathbf h_{k-1}^T, \qquad
  \nabla_{\mathbf b_k} L = \textstyle\sum_{\text{batch}} \nabla_{\mathbf a_k} L,$$

and, unless $k = 1$, the gradient is passed one layer down:

$$\nabla_{\mathbf h_{k-1}} L = W_k^T \nabla_{\mathbf a_k} L, \qquad
  \nabla_{\mathbf a_{k-1}} L = \nabla_{\mathbf h_{k-1}} L \odot f'(\mathbf a_{k-1}).$$

**How the code stores it.** Python lists `W`, `b`, `a`, `h`, indexed by the
**layer number of the lecture**: `W[k]`, `b[k]`, `a[k]` and `h[k]` belong to
layer $k$, `h[0]` is the input and `h[K]` the prediction. `W[0]`, `b[0]` and
`a[0]` are unused (`None`), only so that the indices in the code match the
formulas. The loops run `for k in range(1, K + 1)` forward and
`for k in range(K, 0, -1)` backward.

**The skeleton below gives you the functions, the lists and the loops.** You
fill in the body of each loop — the lines of your Task 3, with `k` in place of
the 1 and the 2. Fill the cells in order; each has step comments. The
functions take the parameters as arguments, so `train_network` must use its
own generator `default_rng(seed)`, never the `rng` of Task 3.

In [ ]:
def init_network(sizes, seed=0, scale=0.1):
    # Returns the lists W and b; W[k], b[k] are the parameters of layer k = 1..K, index 0 is unused.
    gen = np.random.default_rng(seed)
    K = len(sizes) - 1                      # the number of layers
    W = [None] * (K + 1)
    b = [None] * (K + 1)
    for k in range(1, K + 1):
        # Draw W[k], then b[k], each with gen.normal(0.0, scale, shape) and the shape from the
        # table above. In this order, sizes [1, 128, 1] give exactly your Task 3 starting weights.
        # TODO: your code here
        W[k] = np.full((sizes[k], sizes[k - 1]), np.nan)
        b[k] = np.full((sizes[k], 1), np.nan)
    return W, b


def forward_network(W, b, h0):
    # Returns the lists a and h of all layers; h[0] is the input, h[K] the prediction.
    K = len(W) - 1
    a = [None] * (K + 1)
    h = [None] * (K + 1)
    h[0] = h0
    for k in range(1, K + 1):
        # 1. The inner potential a[k] of layer k, from the output h[k-1] of the layer below.
        # 2. The output h[k]: ReLU of a[k] in the hidden layers, a[k] itself in the last layer (k == K).
        # TODO: your code here
        a[k] = np.full((W[k].shape[0], h0.shape[1]), np.nan)
        h[k] = a[k]
    return a, h


def loss_network(W, b, h0, y):
    # The mean loss of the batch, as in Task 3, from the prediction of forward_network.
    # TODO: your code here
    return np.nan

In [ ]:
def backward_network(W, b, h0, y):
    # Returns the lists grad_W and grad_b, with the same indices (and shapes) as W and b.
    K = len(W) - 1
    n = h0.shape[1]
    a, h = forward_network(W, b, h0)
    grad_W = [None] * (K + 1)
    grad_b = [None] * (K + 1)

    # The gradient with respect to the inner potential of the last layer, grad_a
    # (the output layer is linear and the loss is a mean over the batch).
    # TODO: your code here
    grad_a = np.full_like(a[K], np.nan)

    for k in range(K, 0, -1):                          # from the last layer to the first
        # 1. grad_W[k] from grad_a and the input of this layer, h[k-1].
        # 2. grad_b[k]: grad_a summed over the batch (axis=1, keepdims=True).
        # TODO: your code here
        grad_W[k] = np.full_like(W[k], np.nan)
        grad_b[k] = np.full_like(b[k], np.nan)
        if k > 1:
            # 3. Pass the gradient one layer down: back through W[k] to the gradient of h[k-1],
            #    then through the derivative of the ReLU of layer k-1, which gives the new grad_a.
            # TODO: your code here
            grad_a = np.full((W[k].shape[1], n), np.nan)
    return grad_W, grad_b

In [ ]:
def train_network(W, b, X, Y, eta, n_epochs, batch_size=16, seed=0):
    # Trains the lists W and b in place; returns the loss over all samples after every epoch.
    K = len(W) - 1
    gen = np.random.default_rng(seed)
    loss_curve = []
    for epoch in range(n_epochs):
        order = gen.permutation(len(X))
        for start in range(0, len(X), batch_size):
            index = order[start:start + batch_size]
            h0 = X[index].reshape(1, -1)
            y = Y[index].reshape(1, -1)
            # 1. The gradients of this batch with backward_network.
            # 2. A loop over the layers k = 1..K: update W[k] and b[k] (assign into the lists).
            # TODO: your code here
            ...
        loss_curve.append(loss_network(W, b, X.reshape(1, -1), Y.reshape(1, -1)))
    return np.array(loss_curve)

**Three supplied checks.** First, with `sizes = [1, 128, 1]` your loop must be
exactly the Task 3 network: the same starting weights as `W1_start` …
`b2_start`, and after training the same loss curve as your script
(`loss_curve_script`), **bit for bit** — the loop does the same operations in
the same order. Second, every gradient must have the shape of its parameter.
Third, the numerical gradient of 3a on a tiny network with **two** hidden
layers, `[1, 8, 8, 1]`, the first one that passes the gradient through a hidden ReLU into another hidden layer.

In [ ]:
def gradient_check_network(W, b, h0, y, eps=1e-6):
    # Largest difference between your backward_network and the numerical gradient, over every parameter.
    grad_W, grad_b = backward_network(W, b, h0, y)
    differences = []
    for params, grads in [(W, grad_W), (b, grad_b)]:
        for k in range(1, len(params)):
            for index in np.ndindex(params[k].shape):
                original = params[k][index]
                params[k][index] = original + eps
                loss_up = loss_network(W, b, h0, y)
                params[k][index] = original - eps
                loss_down = loss_network(W, b, h0, y)
                params[k][index] = original
                numeric = (loss_up - loss_down) / (2 * eps)
                differences.append(abs(numeric - grads[k][index]))
    return float(np.max(differences))           # nan if anything is unfinished


# 1. [1, 128, 1] against the script of Task 3
W_two, b_two = init_network([1, 128, 1], seed=0)
same_start = all(np.array_equal(mine, script) for mine, script in
                 zip(W_two[1:] + b_two[1:], [W1_start, W2_start, b1_start, b2_start]))
print(f"[1, 128, 1] starts from the weights of Task 3: {same_start}")
curve_layers = train_network(W_two, b_two, X, Y, eta=0.01, n_epochs=300)
print(f"loss after 300 epochs: loop {curve_layers[-1]:.6f}, script {loss_curve_script[-1]:.6f}")
print(f"largest difference between the two loss curves: {np.abs(curve_layers - loss_curve_script).max():.2e}")

# 2. and 3. a tiny network with two hidden layers
W_small, b_small = init_network([1, 8, 8, 1], seed=3, scale=0.8)
h0_small, y_small = X[:6].reshape(1, -1), Y[:6].reshape(1, -1)
grad_W_small, grad_b_small = backward_network(W_small, b_small, h0_small, y_small)
for k in range(1, len(W_small)):
    assert grad_W_small[k].shape == W_small[k].shape, f"grad_W[{k}] has shape {grad_W_small[k].shape}"
    assert grad_b_small[k].shape == b_small[k].shape, f"grad_b[{k}] has shape {grad_b_small[k].shape}"
print("every gradient has the shape of its parameter")
print(f"gradient check on [1, 8, 8, 1]: {gradient_check_network(W_small, b_small, h0_small, y_small):.2e}")

### A deeper network

Now the payoff: a network with two hidden layers is one changed list. Train
`[1, 32, 32, 1]` on the same sine data, with the same settings as before
($\eta = 0.01$, 300 epochs, batches of 16, the default initialization).
**Expected result:** a loss of about 0.057 after 300 epochs. Compare it with
the 0.054 of the two-layer network: is deeper automatically better here? Answer
below the plot.

In [ ]:
# 1. W_deep, b_deep for the sizes [1, 32, 32, 1] with init_network (seed=0).
# 2. Train them with train_network as in the check above; store the loss curve in curve_deep.
# TODO: your code here
W_deep, b_deep = [None], [None]
curve_deep = np.full(300, np.nan)

n_parameters = sum(W_deep[k].size + b_deep[k].size for k in range(1, len(W_deep)))
print(f"[1, 32, 32, 1]: {n_parameters} parameters, loss after 1 epoch {curve_deep[0]:.4f}, "
      f"after 300 epochs {curve_deep[-1]:.4f}")
print(f"[1, 128, 1]   : 385 parameters, loss after 300 epochs {curve_layers[-1]:.4f}")

In [ ]:
grid = np.linspace(X.min(), X.max(), 200).reshape(1, -1)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(X, Y, "*", markersize=4, color="0.6")
axes[0].plot(grid.ravel(), forward_network(W_two, b_two, grid)[1][-1].ravel(), "r-", label="[1, 128, 1]")
axes[0].plot(grid.ravel(), forward_network(W_deep, b_deep, grid)[1][-1].ravel(), "b--", label="[1, 32, 32, 1]")
axes[0].set_ylim(-2.6, 2.6)
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].legend()
axes[1].plot(curve_layers, "r-", label="[1, 128, 1]")
axes[1].plot(curve_deep, "b--", label="[1, 32, 32, 1]")
axes[1].set_yscale("log")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("mean loss")
axes[1].legend()
plt.tight_layout()
plt.show()

*Your answer: is the deeper network better here, and why (not)?*

## Task 5 — From your code to PyTorch

PyTorch does two things that you have just written by hand: it computes the
gradients (**autograd**), and it updates the parameters (an **optimizer**).
Swap one at a time; the loss curve must stay the same through every swap —
then you know exactly what the library does for you. We keep the lists
of Task 4 and the network `[1, 128, 1]`; autograd does not care how many
layers there are.

First a short tour of PyTorch: a few cells to run and read, nothing to write.

### PyTorch in five minutes

**Tensors and NumPy.** A `torch.Tensor` is PyTorch's array, created much like
a NumPy array (`torch.randn`, `torch.zeros`, or `torch.tensor` from a list),
with a `shape` and a `dtype`. The default floating-point type is `float32`,
not NumPy's `float64`. `torch.from_numpy` shares memory with the array and
keeps its `float64`; `torch.tensor(array)` makes a copy, `.numpy()` goes back
and `.float()` converts to `float32`. In this lab we stay in `float64`, so
that PyTorch and your NumPy code agree to the last digit.

In [ ]:
import torch

torch.manual_seed(0)
a = torch.randn(2, 3)                        # normally distributed numbers, like np.random.randn
b = torch.tensor([[1.0, 2.0], [3.0, 4.0]])   # from a list
print("a:", a.shape, a.dtype)
print("b @ b.T =", b @ b.T)                  # @, *, .T and broadcasting work as in NumPy
print("one number out of a tensor:", b[0, 1].item())

x_np = np.linspace(0, 1, 4)
x_shared = torch.from_numpy(x_np)   # shares memory with x_np
x_copy = torch.tensor(x_np)         # an independent copy
print("dtypes:", x_shared.dtype, x_copy.dtype, "  after .float():", x_shared.float().dtype)
x_np[0] = 100.0
print("after changing the array: shared", x_shared[0].item(), "  copy", x_copy[0].item())

**A gradient with autograd.** A tensor created with `requires_grad=True`
records every operation done with it. Reduce the result to a single number —
a scalar loss — and `loss.backward()` stores the derivative of the loss with
respect to the tensor in its `.grad`. Below, the loss is $\sum_i w_i^2$, so
the gradient must be $2 w_i$. `backward()` *adds* to `.grad`: a second
`backward()` doubles it, which is why a training loop sets every `.grad` back
to zero with `.grad.zero_()` after each update.

In [ ]:
w = torch.tensor([1.0, -2.0, 3.0], requires_grad=True)
loss = torch.sum(w ** 2)    # backward() needs a single number
loss.backward()
print("w.grad =", w.grad)
loss = torch.sum(w ** 2)    # backward() frees the record, so the loss is computed again
loss.backward()
print("after a second backward():", w.grad)
w.grad.zero_()
print("after zero_():            ", w.grad)

**Switching the recording off, and the device.** Inside `with torch.no_grad():`
nothing is recorded — used for the parameter update and for evaluation.
`.detach()` returns the same numbers cut out of the record, needed before
`.numpy()`. A tensor lives in RAM (`cpu`) or on a GPU (`cuda`); an operation
needs all its inputs in the same place, `.to(device)` moves a tensor there and
`.cpu()` brings a result back. Today everything runs on the CPU.

In [ ]:
print("w * 5                 : requires_grad =", (w * 5).requires_grad)
with torch.no_grad():
    print("w * 5 inside no_grad(): requires_grad =", (w * 5).requires_grad)
print("as a NumPy array:", w.detach().numpy())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
v = torch.ones(2, 2).to(device)
total = (v * 5).sum()
print(total.device, "->", total.cpu().numpy())

### 5a. Autograd instead of your backward pass

Write `to_torch`, which turns the lists `W`, `b` into lists of tensors that
record their gradients, and `forward_torch` — your `forward_network` for
tensors, with `torch.relu` in place of `np.maximum`, returning only the
prediction. Then compute the loss of the first mini-batch and call
`backward()` on it. The gradients now sit in `W_t[k].grad` and `b_t[k].grad`
— the same numbers your `backward_network` returns for this mini-batch, only
nobody had to write the backward pass.

In [ ]:
def to_torch(W, b):
    # Returns new lists W, b: a float64 tensor copy of every array, recording its gradient; index 0 stays None.
    # TODO: your code here
    return [None] * len(W), [None] * len(b)


def forward_torch(W, b, h0):
    # Your forward pass over the layers with tensors; returns only the prediction.
    # TODO: your code here
    return torch.full((1, h0.shape[1]), float("nan"), dtype=torch.float64)


W_start, b_start = init_network([1, 128, 1], seed=0)       # the starting weights of Task 3
W_t, b_t = to_torch(W_start, b_start)
h0_np, y_np = X[:16].reshape(1, -1), Y[:16].reshape(1, -1)  # the first mini-batch of 3a
h0_t, y_t = torch.tensor(h0_np), torch.tensor(y_np)

# 1. The prediction with forward_torch, then loss_t: the same mean as loss_network, with torch.mean.
# 2. Call backward() on loss_t: autograd fills W_t[1].grad and the others.
# TODO: your code here
loss_t = torch.tensor(float("nan"))

### 5b. Training with autograd

Now train with autograd in place of your `backward_network`. The loop over
the epochs and the mini-batches is the one of your `train_network` and is
given; you write only the lines that change. Two rules from the tour above
enter the loop:

- the **update must not be recorded** — it is not part of the network — so it
  runs inside `with torch.no_grad():`, and it changes the tensor in place
  (`-=`), so that the same tensor, with its `requires_grad`, stays in the list;
- `backward()` **adds** to `.grad`, it does not overwrite it, so after the
  update every `.grad` is set back to zero with `.grad.zero_()`.

**Expected result:** the same loss curve as in Task 4 — the plot after 5c
shows all versions together.

In [ ]:
def train_autograd(W, b, X, Y, eta, n_epochs, batch_size=16, seed=0):
    # train_network with autograd instead of backward_network; returns the loss curve as an array.
    W, b = to_torch(W, b)
    K = len(W) - 1
    gen = np.random.default_rng(seed)
    h0_all, y_all = torch.tensor(X.reshape(1, -1)), torch.tensor(Y.reshape(1, -1))
    loss_curve = []
    for epoch in range(n_epochs):
        order = gen.permutation(len(X))
        for start in range(0, len(X), batch_size):
            index = order[start:start + batch_size]
            h0 = torch.tensor(X[index].reshape(1, -1))
            y = torch.tensor(Y[index].reshape(1, -1))
            # 1. The loss of the batch with forward_torch, then backward() on it.
            # 2. Inside torch.no_grad(), for every layer k: update W[k] and b[k] in place, then zero their .grad.
            # TODO: your code here
            ...
        with torch.no_grad():
            loss_curve.append(float(torch.mean(0.5 * (forward_torch(W, b, h0_all) - y_all) ** 2)))
    return np.array(loss_curve)


W_start, b_start = init_network([1, 128, 1], seed=0)
curve_autograd = train_autograd(W_start, b_start, X, Y, eta=0.01, n_epochs=300)

### 5c. An optimizer instead of your update

The update — `no_grad`, the step, the zeroing — is what an **optimizer**
packages. You hand it all the parameter tensors once, and the batch loop then
always reads

```python
optimizer.zero_grad()   # every .grad back to zero
loss.backward()         # autograd fills every .grad
optimizer.step()        # every parameter takes its step
```

The function below is `train_autograd` again; create the optimizer and write
the batch step with it. Plain `torch.optim.SGD` does exactly your update, so
the curve must not change. (Other optimizers — momentum, Adam — differ only in
what `step()` does; the next lab looks inside them.)

In [ ]:
def train_optimizer(W, b, X, Y, eta, n_epochs, batch_size=16, seed=0):
    # train_autograd with torch.optim.SGD instead of your update; returns the loss curve as an array.
    W, b = to_torch(W, b)
    # Create optimizer: torch.optim.SGD over every tensor of W and b (not the None at index 0), learning rate eta.
    # TODO: your code here
    optimizer = None
    gen = np.random.default_rng(seed)
    h0_all, y_all = torch.tensor(X.reshape(1, -1)), torch.tensor(Y.reshape(1, -1))
    loss_curve = []
    for epoch in range(n_epochs):
        order = gen.permutation(len(X))
        for start in range(0, len(X), batch_size):
            index = order[start:start + batch_size]
            h0 = torch.tensor(X[index].reshape(1, -1))
            y = torch.tensor(Y[index].reshape(1, -1))
            # Every batch: zero the gradients, the loss of the batch, backward, the optimizer step.
            # TODO: your code here
            ...
        with torch.no_grad():
            loss_curve.append(float(torch.mean(0.5 * (forward_torch(W, b, h0_all) - y_all) ** 2)))
    return np.array(loss_curve)


W_start, b_start = init_network([1, 128, 1], seed=0)
curve_optimizer = train_optimizer(W_start, b_start, X, Y, eta=0.01, n_epochs=300)

plt.plot(loss_curve_script, label="Task 3: script")
plt.plot(curve_layers, "--", label="Task 4: loop over the layers")
plt.plot(curve_autograd, ":", label="5b: autograd")
plt.plot(curve_optimizer, "-.", label="5c: optimizer")
plt.yscale("log")
plt.xlabel("epoch")
plt.ylabel("mean loss")
plt.title("four versions of the same network")
plt.legend()
plt.show()

## Task 6 — Your network, deeper, on blood-cell images (compulsory, with an agent)

<p style="color: #c00000;"><strong>! Compulsory hand-in: complete and submit Task 6 within 48 hours after your lab. Attendance will not be credited unless you complete this requirement.</strong></p>

**The practical problem.** A complete blood count with a *differential* tells
a physician how many cells of each type are in a patient's blood. Analysers
photograph single cells from a peripheral blood smear, and a laboratory
technician still checks many of them under the microscope. A classifier that
pre-sorts the cells into types, so that the technician only reviews the
uncertain ones, saves real work. You will build the core of such a classifier
from **your own NumPy network** of Task 4.

**The data: BloodMNIST** (MedMNIST v2, licence CC BY 4.0) — 17,092 microscope
images of individual normal blood cells from the Hospital Clinic of Barcelona
(Acevedo et al., 2020), centre-cropped and downsampled to 28 × 28 RGB pixels,
in **eight classes**: basophil, eosinophil, erythroblast, immature granulocyte,
lymphocyte, monocyte, neutrophil and platelet. The classes are **not
balanced** (from 849 to 2,330 training images). The official split is 11,959
training, 1,712 validation and 3,421 test images; one file of 35 MB:
`https://zenodo.org/records/10519652/files/bloodmnist.npz?download=1`, MD5
`7053d0359d879ad8a5505303e11de1dc`.

The notebook previews four training images of every class (a 66 kB sample,
not the whole file). Some classes are easy to tell apart by eye, others much less
so — and the network sees each image only as 2,352 numbers.

In [ ]:
import io
import urllib.request

BLOOD_CLASSES = ["basophil", "eosinophil", "erythroblast", "immature granulocyte",
                 "lymphocyte", "monocyte", "neutrophil", "platelet"]
url = "https://raw.githubusercontent.com/tomasvicar/MLR-public/master/exercises/data/ex07_bloodmnist_sample.npz"
with urllib.request.urlopen(url) as response:
    sample = np.load(io.BytesIO(response.read()))
images, labels = sample["images"], sample["labels"]
print("images:", images.shape, images.dtype, "  labels:", labels.shape)

fig, axes = plt.subplots(4, 8, figsize=(12, 6.5))
for k, name in enumerate(BLOOD_CLASSES):
    for row, index in enumerate(np.flatnonzero(labels == k)[:4]):
        axes[row, k].imshow(images[index])
        axes[row, k].axis("off")
    axes[0, k].set_title(name.replace(" ", "\n"), fontsize=10)
plt.suptitle("BloodMNIST: four training images per class, 28 × 28 RGB")
plt.tight_layout()
plt.show()

### The goal and what is graded

**The goal:** extend your Task 4 network into a classifier of the eight cell
types, compare a few network sizes on the validation split against a softmax
baseline, evaluate the chosen network once on the test split, and write a
short report. Against Task 4, the input has 28 · 28 · 3 = 2,352 numbers
instead of one, the hidden layers are wider, and the output has eight
neurons with a **softmax** and the **cross-entropy** loss from the lecture.
The backward pass stays your loop over the layers. A network of a few
hundred thousand parameters still trains on a laptop CPU in minutes.

**Writing the instructions for the agent is your job.** There is no
ready-made prompt. Turn the goal and the requirements below into your own
instructions — what to build, in what order, how to check it — and refine
them as you see what the agent does. A good habit: ask for the gradient test
and a quick run on a small part of the data before the full comparison.

**Requirements, and how you check each one in `REVIEW.md`:**

1. **Your own network, extended:** the network code starts from your Task 4
   functions and keeps their conventions (one sample per column, the lists
   `W[k]`, `b[k]`, the loops over the layers, the backward formulas of the
   lecture). → Compare the forward and backward pass in `nn_numpy.py` with
   your Task 4 code: are the conventions, the summed bias and the averaged
   loss still there?
2. **NumPy only** for the network, the training and the optimizer (for
   example Adam from the lecture): no PyTorch, TensorFlow, JAX, autograd
   library or scikit-learn model; scikit-learn may only cross-check a metric.
   → Search every file the agent wrote for `torch`, `tensorflow`, `jax`,
   `autograd`, `sklearn` (your own notebook may use PyTorch in Task 5). Is the
   backward pass really computed in `nn_numpy.py`?
3. **A numerically safe softmax and cross-entropy.** → Is the maximum
   subtracted, is there a `log(0)` risk, is the loss averaged over the batch?
   Feed it the logits (1000, 0, −1000) yourself: the loss must be finite.
4. **The data** are downloaded by a script from the URL above, the MD5 is
   checked, the file is never committed; the official split; standardization
   from the training images only. → Where are the standardization statistics
   computed? Are the images flattened consistently, the labels 0–7 named
   correctly, the class balance reported?
5. **A real gradient test** (`test_gradients.py`, runnable with `pytest`): a
   tiny `float64` network, every analytic gradient, biases included, against a
   central difference with a strict threshold, and a demonstration that it
   fails for a deliberately broken backward pass. → Run it yourself. A check
   that cannot fail checks nothing.
6. **Validation for every choice, test once:** the network size and the epoch
   are chosen on validation; then the test split is used once, for the chosen
   network, the softmax baseline (no hidden layer) and the majority-class
   rule, with accuracy, balanced accuracy, per-class recall and a confusion
   matrix. → Find every use of the test images. Was everything chosen before
   the test set was opened? Is no training accuracy reported as a result?
7. **Reproducible:** fixed seeds, a README with the commands from a fresh
   clone, the measured runtime, and a second run that gives the same
   `results.csv` (or a reported and explained difference). → Inspect the
   seeds, rerun yourself, compare the two `results.csv`, record the command
   and your runtime.
8. **Cautious conclusions** in the report. → Do the learning curves support
   them? How much better than the softmax baseline and the majority rule is
   the network? Which classes are confused? What can this result not
   establish about clinical use?

For each point record in `REVIEW.md` the location (file/line or notebook
cell), what you checked, your finding and how any problem was fixed. "No
problem found" needs evidence too.

**Required files:** `ex07_notebook.ipynb`, `nn_numpy.py`, `download_data.py`,
`train.py`, `test_gradients.py`, `pyproject.toml`, `uv.lock`, `README.md`,
`results.csv` (one row per compared model with its validation accuracy and
balanced accuracy), `test_metrics.json`, `runtime.json`,
`learning_curves.png`, `confusion_matrix.png`, a self-contained `report.html`
and your own `REVIEW.md`. Do not let the agent write `REVIEW.md`.

**In class:** create the repository, invite the teacher, write your first
instructions, and get the gradient test and a short trial run going. **At
home:** the full run, your own review and the hand-in. The setup (your own
notebook in the ZIP folder, the private repository `mlr-ex07`, the invitation
of **MLRteacher**) is described on the exercise page and in the ZIP's
`README.md`.

Submit by pushing within **48 hours after the end of your lab**. What is pushed
before the deadline counts. Hand-ins are reviewed automatically by AI.

<p style="color: #c00000;"><strong>! Plagiarism check: all hand-ins will be checked for plagiarism.</strong></p>

Without a working agent, meet the same requirements yourself at home.
*Optional after the required run:* L2 weight decay or dropout from the
lecture, chosen on validation only.

**What to expect:** the softmax baseline reaches roughly 0.8 validation
accuracy and a NumPy network with hidden layers clearly more; the whole
comparison takes minutes, not hours, on a laptop CPU. Much better or much
worse results deserve an investigation, not automatic acceptance.

## Summary

* A neuron is $\hat y = f(\mathbf w^T \mathbf x + b)$; its gradient always
  splits into loss $\times$ activation $\times$ input.
* Backpropagation is the chain rule written once per layer:
  $\nabla_{\mathbf a_k} L = \nabla_{\mathbf h_k} L \odot f'(\mathbf a_k)$,
  $\nabla_{\mathbf h_{k-1}} L = W_k^T \nabla_{\mathbf a_k} L$,
  $\nabla_{W_k} L = \nabla_{\mathbf a_k} L\, \mathbf h_{k-1}^T$.
* The shape of a gradient equals the shape of what it is a gradient of. Use
  that as your first check, and the **numerical gradient** as your second.
* A refactoring, or a switch to a library, is checked by **getting the same
  numbers**: your script, your loop over the layers, autograd and
  `torch.optim.SGD` gave the same loss curve. `loss.backward()` computes what you wrote in Task 3;
  an optimizer performs your update.
* A network of any depth is the same recipe in two loops over the layers:
  forward from the first layer to the last, backward from the last to the
  first. With two layers the loop gives exactly your script; a deeper network
  is just a longer list of sizes.

Next time we continue in **PyTorch** from exactly this point: what autograd
records, what optimizers such as Adam keep in memory, `nn.Module` for networks of any size, and a
convolutional network on images.